<a href="https://colab.research.google.com/github/niharika-saha/Temporal-Trio_Multimodal-Fine-Tuning-with-SLM/blob/blip/nlp_orange_blip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets accelerate peft bitsandbytes torchvision --quiet
!pip install huggingface_hub --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.8 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from datasets import load_dataset

dataset = load_dataset("rootsautomation/RICO-Screen2Words")

print(dataset)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00008.parquet:   0%|          | 0.00/216M [00:00<?, ?B/s]

data/train-00001-of-00008.parquet:   0%|          | 0.00/219M [00:00<?, ?B/s]

data/train-00002-of-00008.parquet:   0%|          | 0.00/225M [00:00<?, ?B/s]

data/train-00003-of-00008.parquet:   0%|          | 0.00/215M [00:00<?, ?B/s]

...

data/test-00001-of-00002.parquet:   0%|          | 0.00/244M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15743 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/2364 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4310 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['screenId', 'captions', 'file_name', 'app_package_name', 'play_store_name', 'category', 'average_rating', 'number_of_ratings', 'number_of_downloads', 'file_name_icon', 'file_name_semantic', 'semantic_annotations', 'view_hierarchy', 'image', 'image_icon', 'image_semantic'],
        num_rows: 15743
    })
    val: Dataset({
        features: ['screenId', 'captions', 'file_name', 'app_package_name', 'play_store_name', 'category', 'average_rating', 'number_of_ratings', 'number_of_downloads', 'file_name_icon', 'file_name_semantic', 'semantic_annotations', 'view_hierarchy', 'image', 'image_icon', 'image_semantic'],
        num_rows: 2364
    })
    test: Dataset({
        features: ['screenId', 'captions', 'file_name', 'app_package_name', 'play_store_name', 'category', 'average_rating', 'number_of_ratings', 'number_of_downloads', 'file_name_icon', 'file_name_semantic', 'semantic_annotations', 'view_hierarchy', 'image', 'image_icon', 'imag

In [ ]:
#load blip model
from transformers import BlipProcessor, BlipForConditionalGeneration

processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

In [ ]:
#check
sample = dataset["train"][0]

print(sample["captions"])
print(type(sample["image"]))#

['notification displaying a calendar with done option', 'pop up displaying a calendar', 'pop up displaying calendar to select date', 'pop up window of calendar with options', 'pop-up of a calendar to select date of birth']
<class 'PIL.JpegImagePlugin.JpegImageFile'>


In [ ]:
#test with small subset
train_dataset = dataset["train"].select(range(2000))
val_dataset = dataset["val"].select(range(500))

In [ ]:
def preprocess(example):

    image = example["image"]
    caption = example["captions"][0]

    inputs = processor(
        images=image,
        text=caption,
        padding="max_length",
        truncation=True
    )

    inputs["labels"] = inputs["input_ids"]

    return inputs

In [ ]:
train_dataset = train_dataset.map(
    preprocess,
    remove_columns=train_dataset.column_names
)

val_dataset = val_dataset.map(
    preprocess,
    remove_columns=val_dataset.column_names
)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
train_dataset.set_format("torch")
val_dataset.set_format("torch")

In [ ]:
#training arguments
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./rico-caption-model",
    per_device_train_batch_size=8,
    num_train_epochs=3,
    learning_rate=5e-5,
    logging_steps=100,
    save_steps=500,
    fp16=True
)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./rico-caption-model",
    per_device_train_batch_size=1,  # Further reduced batch size to 1 to prevent OutOfMemoryError
    num_train_epochs=3,
    learning_rate=5e-5,
    logging_steps=100,
    save_steps=500,
    fp16=True
)

In [ ]:
import torch

def collate_fn(batch):

    pixel_values = torch.stack([item["pixel_values"].squeeze(0) for item in batch])
    input_ids = torch.stack([item["input_ids"] for item in batch])
    attention_mask = torch.stack([item["attention_mask"] for item in batch])
    labels = torch.stack([item["labels"] for item in batch])

    return {
        "pixel_values": pixel_values,
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn
)

In [ ]:
#train
trainer.train()

trainer.save_model("rico-ui-caption-model")
processor.save_pretrained("rico-ui-caption-model")

Step,Training Loss
100,4.164329
200,0.068696
300,0.057105
400,0.051559
500,0.048946
600,0.051416
700,0.052526
800,0.050438
900,0.051112
1000,0.048226


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6000, training_loss=0.10728435512383779, metrics={'train_runtime': 2087.4414, 'train_samples_per_second': 2.874, 'train_steps_per_second': 2.874, 'total_flos': 3.561028015620096e+18, 'train_loss': 0.10728435512383779, 'epoch': 3.0})

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration

processor = BlipProcessor.from_pretrained("rico-ui-caption-model")
model = BlipForConditionalGeneration.from_pretrained("rico-ui-caption-model")


model.push_to_hub("rico-screen2words-blip-caption")
processor.push_to_hub("rico-screen2words-blip-caption")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

['rico-ui-caption-model/processor_config.json']

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image

processor = BlipProcessor.from_pretrained("rico-screen2words-blip-caption")
model = BlipForConditionalGeneration.from_pretrained("rico-screen2words-blip-caption")

image = dataset["test"][0]["image"]

inputs = processor(image, return_tensors="pt" )

out = model.generate(**inputs)

print(processor.decode(out[0], skip_special_tokens=True))

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("rico-ui-caption-model", "zip", "rico-ui-caption-model")
files.download("rico-ui-caption-model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("rico-caption-model", "zip", "rico-caption-model")
files.download("rico-caption-model.zip")

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration

processor = BlipProcessor.from_pretrained("rico-ui-caption-model")
model = BlipForConditionalGeneration.from_pretrained("rico-ui-caption-model")

model.push_to_hub("rico-screen2words-blip-caption")
processor.push_to_hub("rico-screen2words-blip-caption")

RepositoryNotFoundError: 404 Client Error. (Request ID: Root=1-69b163cc-0fda500e0d6ae7a4214b1492;ed4800b8-f328-4cfe-9b8e-7f105d27f69f)

Repository Not Found for url: https://huggingface.co/api/models/rico-ui-caption-model/tree/main/additional_chat_templates?recursive=false&expand=false.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication